### Clean ACL/OCL Papers (1979-2017)

This notebook removes the overlap of ACL/OCL anthology papers from 1979 to 2017 with anthology splits.
- Loads entire ACL anthology metadata and a 1979-2017 paper subset.
- Removes rows with missing abstract or title.
- Checks dataset overlaps by title and abstract.
- Cleans and filters the data.
- Saves it back to splits/

In [1]:
import bibtexparser
import numpy as np
import pandas as pd

from bibtexparser.bparser import BibTexParser
from pathlib import Path
import gzip
import re
import unicodedata
import os
from tqdm import tqdm

In [2]:
all_anthology = pd.read_csv("splits/all_anthology_data.csv")
all_anthology.head()

,Unnamed: 0,key,year,title,abstract
0,0,lee-etal-2021-paraphrasing,2021,Paraphrasing Compound Nominalizations,A nominalization uses a deverbal noun to descr...
1,1,zhang-etal-2022-cross,2022,Cross-Modal Similarity-Based Curriculum Learni...,Image captioning models require the high-level...
2,2,huang-etal-2021-disentangling,2021,Disentangling Semantics and Syntax in Sentence...,Pre-trained language models have achieved huge...
3,3,ma-etal-2020-simple,2020,A Simple and Effective Unified Encoder for Doc...,Most of the existing models for document-level...
4,4,gou-etal-2023-mvp,2023,MvP: Multi-view Prompting Improves Aspect Sent...,Generative methods greatly promote aspect-base...


In [3]:
df = pd.read_csv("splits/acl_ocl_papers_1979_2017.csv")
df.head()

,title,abstract,venue,year,authors
0,A Snapshot of KDS: A Knowledge Delivery System,KDS Is a computer program which creates multl-...,ACL,1979,James A. Moore; William C. Mann
1,An Application of Automated Language Understan...,NaN,ACL,1979,Georgette Silva; Christine Montgomery; Don Dwi...
2,Applications,"Truth, like beauty, is in the eye of the behol...",ACL,1979,David G. Hays
3,Design for Dialogue Comprehension,This paper describes aspects of the design of ...,ACL,1979,William C. Mann
4,Discourse: Codes and Clues in Contexts,NaN,ACL,1979,Jane J. Robinson


In [4]:
print(f"Number of rows before removing NaN: {len(df)}")
df = df.dropna(subset=["title", "abstract", "year"]).reset_index(drop=True)
print(f"Number of rows before removing NaN: {len(df)}")

Number of rows before removing NaN: 9801
Number of rows before removing NaN: 9489


In [5]:
matching_titles_count = df['title'].isin(all_anthology['title']).sum()
matching_abstract_count = df['abstract'].isin(all_anthology['abstract']).sum()
print(matching_titles_count, matching_abstract_count)

1062 379


In [6]:
common_titles = set(df['title']).intersection(set(all_anthology['title']))
common_abstracts = set(df['abstract']).intersection(set(all_anthology['abstract']))

for abstract in common_abstracts:
    assert df[df['abstract'] == abstract].iloc[0]['title'] == all_anthology[all_anthology['abstract'] == abstract].iloc[0]['title']

common_abstract_titles = set(df[df['abstract'].isin(common_abstracts)]['title'])

assert len(common_abstract_titles - common_titles) == 0

years = set(df[df['title'].isin(common_titles)]['year'])

print(f"Common titles: {len(common_titles)}")
print(f"Common abstracts: {len(common_abstracts)}")
print(f"Common paper years: {years}")

Common titles: 1062
Common abstracts: 379
Common paper years: {2016, 2017, 1987, 1991, 1993, 2015, 2013, 2014, 1983}


In [7]:
df = df[(~df['title'].isin(common_titles)) & (~df['abstract'].isin(common_abstracts))].reset_index(drop=True)
print(f"Number of rows after dropping common rows: {len(df)}")

Number of rows after dropping common rows: 8427


In [8]:
common_titles = set(df['title']).intersection(set(all_anthology['title']))
common_abstracts = set(df['abstract']).intersection(set(all_anthology['abstract']))

for abstract in common_abstracts:
    assert df[df['abstract'] == abstract].iloc[0]['title'] == all_anthology[all_anthology['abstract'] == abstract].iloc[0]['title']

common_abstract_titles = set(df[df['abstract'].isin(common_abstracts)]['title'])

assert len(common_abstract_titles - common_titles) == 0

years = set(df[df['title'].isin(common_titles)]['year'])

print("AFTER CLEANING")
print(f"Common titles: {len(common_titles)}")
print(f"Common abstracts: {len(common_abstracts)}")
print(f"Common paper years: {years}")

AFTER CLEANING
Common titles: 0
Common abstracts: 0
Common paper years: set()


In [9]:
header_map = {
    'title': "Title",
    "abstract": "Abstract",
    "year": "Year",
    "venue": "Venue",
    "authors": "Authors"
}

df = df.rename(columns=header_map)
df['ID'] = df.reset_index().index

In [10]:
df.head()

,Title,Abstract,Venue,Year,Authors,ID
0,A Snapshot of KDS: A Knowledge Delivery System,KDS Is a computer program which creates multl-...,ACL,1979,James A. Moore; William C. Mann,0
1,Applications,"Truth, like beauty, is in the eye of the behol...",ACL,1979,David G. Hays,1
2,Design for Dialogue Comprehension,This paper describes aspects of the design of ...,ACL,1979,William C. Mann,2
3,Knowledge Organization and Application: Brief ...,My brief comments on the papers in this sessio...,ACL,1979,Aravind K. Joshi,3
4,Natural Language Input to a Computer-Based Gla...,"A ""Front End"" for a Computer-Based Glaucoma Co...",ACL,1979,Victor B. Ciesielski,4


In [11]:
df.to_csv("splits/acl_ocl_papers_1979_2017_cleaned.csv", index=False)

## Further removing incomplete abstracts

In [12]:
problematic_abstracts = pd.read_csv("splits/problematic_abstracts.csv")
with open("splits/problematic_row_ids.txt", "r") as f:
    problematic_row_ids = [int(line.strip()) for line in f if line.strip().isdigit()]

In [13]:
problematic_df = df[df['ID'].isin(problematic_row_ids)]
remaining = df[~df['ID'].isin(problematic_row_ids)].reset_index(drop=True)

In [15]:
len(remaining)

8201

In [14]:
problematic_df.to_csv("splits/acl_ocl_papers_1979_2017_problematic.csv", index=False)
remaining.to_csv("splits/acl_ocl_papers_1979_2017_final.csv", index=False)